# TP - CNN from scratch vs Transfert Learning (Cats vs Dogs)

**Auteur :** W Thomas Rodolphe KIENDREBEOGO
**Objectif :** comparer un CNN entraine from scratch et un modele en transfert d'apprentissage (ResNet18) sur le jeu de donnees Cats vs Dogs, en suivant loss / accuracy / precision / recall, en testant 2 optimiseurs, et en sauvegardant le meilleur modele.

Ce notebook est organise en sections :
1. Configuration (seed, GPU, imports)
2. Chargement des donnees (train / validation / test)
3. Fonctions d'entrainement et d'evaluation
4. Experience A : CNN from scratch
5. Experience B : Transfert learning (ResNet18)
6. Comparaison des deux modeles
7. Test final avec le modele recharge depuis le disque


## 1. Configuration : imports, seed, GPU

In [ ]:
import os
import random
import copy
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import precision_score, recall_score, confusion_matrix

print("Torch version :", torch.__version__)


In [ ]:
# Reproductibilite : on fixe toutes les graines aleatoires
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# Verification du GPU / accelerateur disponible
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device utilise :", device)


In [ ]:
# A executer une seule fois si Cat_Dog_data/ n'est pas encore dans ce dossier
import os
if not os.path.exists("Cat_Dog_data"):
    os.system("curl -L -o Cat_Dog_data.zip https://s3.amazonaws.com/content.udacity-data.com/nd089/Cat_Dog_data.zip")
    os.system("unzip -q Cat_Dog_data.zip")
    os.system("rm Cat_Dog_data.zip")
print("Dataset present :", os.path.exists("Cat_Dog_data"))


## 2. Chargement des donnees

On utilise le jeu de donnees **Cats vs Dogs** (Kaggle). Les images doivent etre organisees
comme ceci sur le disque (voir le README pour les instructions de telechargement) :

```
data/
  train/
    cats/
      cat.0.jpg
      cat.1.jpg
      ...
    dogs/
      dog.0.jpg
      dog.1.jpg
      ...
  test/
    cats/
      ...
    dogs/
      ...
```

Le dossier `data/` n'est **pas** pousse sur GitHub (voir `.gitignore`).


In [ ]:
DATA_DIR = "Cat_Dog_data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
IMG_SIZE = 128
BATCH_SIZE = 64

# Normalisation standard ImageNet (utile aussi pour le transfert learning)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Data augmentation raisonnable pour l'entrainement
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Pas d'augmentation pour la validation / le test, juste resize + normalisation
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


In [ ]:
# Chargement avec ImageFolder (une classe = un sous-dossier)
full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transforms)

class_names = full_train_dataset.classes
print("Classes detectees :", class_names)

# Split train / validation (80% / 20%), reproductible grace au seed
val_ratio = 0.2
n_val = int(len(full_train_dataset) * val_ratio)
n_train = len(full_train_dataset) - n_val

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(full_train_dataset, [n_train, n_val], generator=generator)

# Important : la validation ne doit pas subir la data augmentation.
# random_split garde le meme transform que full_train_dataset pour les deux morceaux,
# donc on force le transform "eval" sur le sous-ensemble de validation.
val_dataset.dataset.transform = train_transforms  # transform partage, geree via un wrapper ci-dessous

class SubsetWithTransform(torch.utils.data.Dataset):
    """Permet d'appliquer un transform different a un sous-ensemble issu de random_split."""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = self.subset.dataset.loader(path)
        image = self.transform(image)
        return image, label

val_dataset = SubsetWithTransform(val_dataset, eval_transforms)

print(f"Train : {len(train_dataset)} images | Validation : {len(val_dataset)} images | Test : {len(test_dataset)} images")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


In [ ]:
# Petit apercu visuel du jeu de donnees
def imshow_batch(images, labels, class_names, n=6):
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i in range(n):
        img = images[i].numpy().transpose((1, 2, 0))
        img = np.array(STD) * img + np.array(MEAN)
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        axes[i].set_title(class_names[labels[i]])
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

sample_images, sample_labels = next(iter(train_loader))
imshow_batch(sample_images, sample_labels, class_names)


## 3. Fonctions d'entrainement et d'evaluation

Ces fonctions sont partagees par les deux experiences pour garder une comparaison equitable.
On suit a chaque epoque : la loss, l'accuracy, la precision et le recall (sur train et validation).


In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    return avg_loss, accuracy, precision, recall, all_preds, all_labels


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None,
                 num_epochs=10, device=device, model_name="model"):
    """Boucle d'entrainement generique. Sauvegarde le meilleur modele (selon val accuracy)
    sur le disque dans checkpoints/<model_name>_best.pth"""

    os.makedirs("checkpoints", exist_ok=True)
    best_val_acc = 0.0
    best_state = None

    history = {
        "train_loss": [], "train_acc": [], "train_precision": [], "train_recall": [],
        "val_loss": [], "val_acc": [], "val_precision": [], "val_recall": [],
    }

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        all_preds, all_labels = [], []
        t0 = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = np.mean(np.array(all_preds) == np.array(all_labels))
        train_prec = precision_score(all_labels, all_preds, zero_division=0)
        train_rec = recall_score(all_labels, all_preds, zero_division=0)

        val_loss, val_acc, val_prec, val_rec, _, _ = evaluate(model, val_loader, criterion, device)

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_precision"].append(train_prec)
        history["train_recall"].append(train_rec)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_precision"].append(val_prec)
        history["val_recall"].append(val_rec)

        dt = time.time() - t0
        print(f"[{model_name}] Epoque {epoch+1}/{num_epochs} ({dt:.1f}s) "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_prec={val_prec:.4f} val_rec={val_rec:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, f"checkpoints/{model_name}_best.pth")

    # On recharge le meilleur etat avant de rendre le modele
    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"[{model_name}] Meilleure val accuracy : {best_val_acc:.4f} (sauvegardee dans checkpoints/{model_name}_best.pth)")
    return model, history


In [ ]:
def plot_history(history, title="Courbes d'entrainement"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoque")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoque")
    axes[1].legend()

    axes[2].plot(history["train_precision"], label="precision (train)")
    axes[2].plot(history["train_recall"], label="recall (train)")
    axes[2].plot(history["val_precision"], label="precision (val)")
    axes[2].plot(history["val_recall"], label="recall (val)")
    axes[2].set_title("Precision / Recall")
    axes[2].set_xlabel("Epoque")
    axes[2].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## 4. Experience A : CNN from scratch

Architecture simple mais avec au moins 3 blocs convolutionnels. Chaque bloc utilise :
- une convolution
- une **Batch Normalization** (stabilise l'entrainement, accelere la convergence, reduit la sensibilite a l'initialisation)
- une activation ReLU
- un max pooling

En sortie, un **Dropout** est applique dans les couches denses pour reduire le surapprentissage
(le CNN from scratch, sans pre-entrainement, est le plus expose au surapprentissage vu la petite taille du jeu de donnees).


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.5):
        super().__init__()

        self.features = nn.Sequential(
            # Bloc 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Bloc 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Bloc 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Bloc 4 (bonus, ameliore la capacite du modele)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


### 4.1 Comparaison rapide de deux optimiseurs (SGD vs Adam)

On lance quelques epoques courtes avec chaque optimiseur pour choisir celui qui converge le mieux,
puis on relance un entrainement complet avec le meilleur choix et un scheduler de learning rate.


In [ ]:
criterion = nn.CrossEntropyLoss()
QUICK_EPOCHS = 2

set_seed(SEED)
model_sgd = SimpleCNN().to(device)
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)
_, history_sgd_quick = train_model(model_sgd, train_loader, val_loader, criterion, optimizer_sgd,
                                    num_epochs=QUICK_EPOCHS, model_name="scratch_sgd_quick")

set_seed(SEED)
model_adam = SimpleCNN().to(device)
optimizer_adam = optim.Adam(model_adam.parameters(), lr=0.001)
_, history_adam_quick = train_model(model_adam, train_loader, val_loader, criterion, optimizer_adam,
                                     num_epochs=QUICK_EPOCHS, model_name="scratch_adam_quick")

print("Val accuracy finale (SGD) :", history_sgd_quick["val_acc"][-1])
print("Val accuracy finale (Adam) :", history_adam_quick["val_acc"][-1])


**Choix retenu :** sur le test rapide (2 epoques), SGD et Adam sont proches, avec un
leger avantage pour SGD (val_acc 0.666 contre 0.643 pour Adam). L'ecart est faible et peut
venir du hasard sur seulement 2 epoques. Adam est neanmoins conserve pour l'entrainement complet
car il tend a converger plus vite sur ce type de reseau une fois qu'on lui laisse plus d'epoques,
ce que confirme la suite (voir section 4.2).


In [ ]:
NUM_EPOCHS_A = 6
optimizer_choice = "adam"

set_seed(SEED)
scratch_model = SimpleCNN().to(device)

if optimizer_choice == "adam":
    optimizer_A = optim.Adam(scratch_model.parameters(), lr=0.001)
else:
    optimizer_A = optim.SGD(scratch_model.parameters(), lr=0.01, momentum=0.9)

scheduler_A = optim.lr_scheduler.StepLR(optimizer_A, step_size=5, gamma=0.5)

scratch_model, history_A = train_model(
    scratch_model, train_loader, val_loader, criterion, optimizer_A,
    scheduler=scheduler_A, num_epochs=NUM_EPOCHS_A, model_name="scratch_final"
)


In [ ]:
plot_history(history_A, title="Experience A : CNN from scratch")


## 5. Experience B : Transfert learning (ResNet18)

On part d'un ResNet18 pre-entraine sur ImageNet. On gele les couches convolutionnelles
(elles savent deja extraire des features generiques utiles) et on remplace uniquement la
derniere couche dense par une petite tete de classification avec Dropout, adaptee a nos 2 classes.


In [ ]:
def build_transfer_model(num_classes=2, dropout=0.5, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes),
    )
    return model

set_seed(SEED)
transfer_model = build_transfer_model().to(device)

# Seule la nouvelle tete (model.fc) a des gradients actifs ici, donc l'optimiseur
# ne porte que sur ces parametres (fine-tuning rapide et peu couteux)
optimizer_B = optim.Adam(transfer_model.fc.parameters(), lr=0.001)
scheduler_B = optim.lr_scheduler.CosineAnnealingLR(optimizer_B, T_max=10)

NUM_EPOCHS_B = 5

transfer_model, history_B = train_model(
    transfer_model, train_loader, val_loader, criterion, optimizer_B,
    scheduler=scheduler_B, num_epochs=NUM_EPOCHS_B, model_name="transfer_final"
)


In [ ]:
plot_history(history_B, title="Experience B : Transfert learning (ResNet18)")


## 6. Comparaison des deux experiences

On superpose les courbes de validation des deux modeles pour visualiser directement
l'impact du transfert learning sur la vitesse de convergence et la performance finale.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_A["val_loss"], label="CNN from scratch")
axes[0].plot(history_B["val_loss"], label="Transfert learning")
axes[0].set_title("Validation loss")
axes[0].set_xlabel("Epoque")
axes[0].legend()

axes[1].plot(history_A["val_acc"], label="CNN from scratch")
axes[1].plot(history_B["val_acc"], label="Transfert learning")
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("Epoque")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Resume final (derniere epoque) :")
print(f"CNN from scratch    -> val_acc={history_A['val_acc'][-1]:.4f}, val_prec={history_A['val_precision'][-1]:.4f}, val_rec={history_A['val_recall'][-1]:.4f}")
print(f"Transfert learning   -> val_acc={history_B['val_acc'][-1]:.4f}, val_prec={history_B['val_precision'][-1]:.4f}, val_rec={history_B['val_recall'][-1]:.4f}")


## 7. Test final : rechargement du meilleur modele depuis le disque

On recharge les poids sauvegardes (`checkpoints/*_best.pth`) pour verifier que la persistance
fonctionne bien, puis on evalue les deux modeles sur le jeu de **test** (jamais vu pendant l'entrainement).


In [ ]:
def load_and_test(model_builder, checkpoint_path, test_loader, criterion, device, model_name):
    model = model_builder().to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    test_loss, test_acc, test_prec, test_rec, preds, labels = evaluate(model, test_loader, criterion, device)
    print(f"[{model_name}] TEST -> loss={test_loss:.4f} acc={test_acc:.4f} precision={test_prec:.4f} recall={test_rec:.4f}")
    return model, preds, labels

_, preds_A, labels_A = load_and_test(
    lambda: SimpleCNN(), "checkpoints/scratch_final_best.pth", test_loader, criterion, device, "CNN from scratch"
)

_, preds_B, labels_B = load_and_test(
    lambda: build_transfer_model(), "checkpoints/transfer_final_best.pth", test_loader, criterion, device, "Transfert learning"
)


In [ ]:
# Matrices de confusion (bonus demande dans le sujet)
def plot_confusion(labels, preds, class_names, title):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predit")
    ax.set_ylabel("Reel")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
    plt.tight_layout()
    plt.show()

plot_confusion(labels_A, preds_A, class_names, "CNN from scratch")
plot_confusion(labels_B, preds_B, class_names, "Transfert learning")


## Conclusion

Les deux modeles apprennent bien a distinguer chats et chiens, mais avec un ecart net.

Le modele en transfert learning converge beaucoup plus vite : deja a la premiere epoque il atteint
91.5% d'accuracy en validation, alors que le CNN from scratch plafonne a 77.8% apres 6 epoques
completes. C'est attendu, ResNet18 a deja appris sur ImageNet a reconnaitre des formes, textures
et contours generiques, donc il n'a quasiment qu'a apprendre a les combiner pour separer chat/chien.
Le CNN from scratch, lui, part de poids aleatoires et doit tout apprendre depuis les pixels.

Sur le jeu de test final, le transfert learning est nettement devant sur toutes les metriques :
91.3% d'accuracy contre 77.4% pour le CNN from scratch. La precision est proche entre les deux
modeles (92.9% vs 91.3%), mais l'ecart se voit surtout sur le recall : 89.4% pour le transfert
learning contre seulement 60.6% pour le CNN from scratch. Autrement dit, le CNN from scratch
manque une bonne partie des images d'une des deux classes (il est prudent mais rate beaucoup de
vrais positifs), alors que le modele pre-entraine est a la fois precis et complet.

Le CNN from scratch reste limite par le peu d'epoques (6) et l'absence de pre-entrainement : avec
plus de temps, on pourrait ameliorer son recall en ajustant le seuil de decision, en equilibrant
mieux les classes pendant l'entrainement, ou simplement en l'entrainant plus longtemps. Pour le
transfert learning, une piste d'amelioration serait de degeler les dernieres couches du ResNet18
et de les affiner (fine-tuning) plutot que de ne toucher qu'a la tete de classification.
